In [9]:
# Install LiteEFG
!pip install LiteEFG
# Other utilities
!pip install numpy pandas tqdm

In [1]:
# Write the .game file to disk
game_content = """# Simple 2-depth Coin Matching Game
#
# Two players each have a coin. Player 1 places their coin (Heads or Tails),
# then Player 2 responds. If they match, Player 2 wins; if not, Player 1 wins.
#
# Opt {
#     game_tree: true,
#     num_players: 2,
#     depth: 2,
#     output_file: \"GameInstances/coin_match.txt\",
# }
#
node / player 1 actions H T
node /P1:H player 2 actions H T
node /P1:H/P2:H leaf payoffs 1=-1 2=1
node /P1:H/P2:T leaf payoffs 1=1 2=-1
node /P1:T player 2 actions H T
node /P1:T/P2:H leaf payoffs 1=1 2=-1
node /P1:T/P2:T leaf payoffs 1=-1 2=1
infoset pl1_0__ nodes /
infoset pl2_0__H nodes /P1:H
infoset pl2_1__T nodes /P1:T
"""

game_file_path = "coin_match.game"
with open(game_file_path, "w") as f:
    f.write(game_content)
print(f"Game file written to: {game_file_path}")

Game file written to: matching_pennies.game


In [11]:
import LiteEFG as efg
from LiteEFG.baselines.CFR import graph as CFR   # correct class name is 'graph'
import numpy as np
import pandas as pd
from tqdm import trange

In [ ]:
# Load the game

env = efg.FileEnv(game_file_path, traverse_type="Enumerate")
print("Game loaded successfully!")

In [5]:
# Instantiate CFR using the correct 'graph' class from the installed source
alg = CFR()

# Register the computation graph with the environment
env.set_graph(alg)
print("CFR graph registered.")

===============Graph is ready for CFR===============


CFR graph registered.


In [6]:
# Run CFR for 1000 iterations, logging exploitability every 100 steps
NUM_ITER   = 1000
PRINT_FREQ = 100
history    = []

for i in trange(NUM_ITER, desc="CFR"):
    alg.update_graph(env)
    env.update_strategy(alg.current_strategy())

    if i % PRINT_FREQ == 0 or i == NUM_ITER - 1:
        explo = env.exploitability(alg.current_strategy(), "avg-iterate")
        total = sum(explo)
        history.append({"iteration": i, "exploitability": round(total, 6)})

print("\nDone.")

CFR: 100%|██████████████████████████████| 1000/1000 [00:00<00:00, 371604.86it/s]


Done.


In [7]:
# Show exploitability convergence — should decrease toward 0.0
df_history = pd.DataFrame(history)
print(df_history.to_string(index=False))

 iteration  exploitability
         0             0.0
       100             0.0
       200             0.0
       300             0.0
       400             0.0
       500             0.0
       600             0.0
       700             0.0
       800             0.0
       900             0.0
       999             0.0


In [8]:
# Inspect final per-infoset strategies (LiteEFG uses 1-indexed player IDs)
print("=" * 50)
for player_id in range(1, 3):
    print(f"\nPlayer {player_id} strategy (avg-iterate):")
    strategy_pairs = env.get_strategy(player_id, alg.current_strategy(), "avg-iterate")
    rows = []
    for infoset_name, action_probs in strategy_pairs:
        rows.append({"Infoset": infoset_name, "Action Probs": list(action_probs)})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
print("=" * 50)
print("\nBoth players should converge to [0.5, 0.5] — the Nash Equilibrium for Matching Pennies.")


Player 1 strategy (avg-iterate):
Infoset Action Probs
pl1_0__   [0.5, 0.5]

Player 2 strategy (avg-iterate):
 Infoset Action Probs
pl2_0__H   [1.0, 0.0]
pl2_1__T   [0.0, 1.0]

Both players should converge to [0.5, 0.5] — the Nash Equilibrium for Matching Pennies.


## A new game

In [11]:
# New game: Blind Matching Pennies (Imperfect Information)
# Same payoffs as before, but Player 2 cannot observe Player 1's move.
# P2's two infosets are MERGED into one — this is what makes it imperfect information.

game_content_ii = """# Blind Matching Pennies — Imperfect Information EFG
#
# Player 1 chooses H or T secretly.
# Player 2 must respond WITHOUT knowing P1's choice.
# Merging /P1:H and /P1:T into one infoset for P2 models this blindness.
#
# Opt {
#     game_tree: true,
#     num_players: 2,
#     depth: 2,
# }
#
node / player 1 actions H T
node /P1:H player 2 actions H T
node /P1:H/P2:H leaf payoffs 1=-1 2=1
node /P1:H/P2:T leaf payoffs 1=1 2=-1
node /P1:T player 2 actions H T
node /P1:T/P2:H leaf payoffs 1=1 2=-1
node /P1:T/P2:T leaf payoffs 1=-1 2=1
infoset pl1_0__ nodes /
infoset pl2_0__ nodes /P1:H /P1:T
"""
# Note: pl2 now has ONE infoset covering BOTH /P1:H and /P1:T
# In the perfect-info version we had two separate infosets: pl2_0__H and pl2_1__T

ii_game_file_path = "blind_coin_match.game"
with open(ii_game_file_path, "w") as f:
    f.write(game_content_ii)
print(f"IIEFG written to: {ii_game_file_path}")

# Load, solve with CFR, report
env_ii = efg.FileEnv(ii_game_file_path, traverse_type="Enumerate")
alg_ii = CFR()
env_ii.set_graph(alg_ii)

NUM_ITER   = 1000
PRINT_FREQ = 100
history_ii = []

for i in trange(NUM_ITER, desc="CFR (IIEFG)"):
    alg_ii.update_graph(env_ii)
    env_ii.update_strategy(alg_ii.current_strategy())

    if i % PRINT_FREQ == 0 or i == NUM_ITER - 1:
        explo = env_ii.exploitability(alg_ii.current_strategy(), "avg-iterate")
        total = sum(explo)
        history_ii.append({"iteration": i, "exploitability": round(total, 6)})

print("\nDone.\n")
df_ii = pd.DataFrame(history_ii)
print(df_ii.to_string(index=False))

print("\n" + "=" * 50)
for player_id in range(1, 3):
    print(f"\nPlayer {player_id} strategy (avg-iterate):")
    strategy_pairs = env_ii.get_strategy(player_id, alg_ii.current_strategy(), "avg-iterate")
    rows = [{"Infoset": name, "Action Probs": list(probs)} for name, probs in strategy_pairs]
    print(pd.DataFrame(rows).to_string(index=False))
print("=" * 50)
print("\nP2 has ONE infoset — same [0.5, 0.5] mix regardless of what P1 did.")

IIEFG written to: blind_coin_match.game
===============Graph is ready for CFR===============




CFR (IIEFG): 100%|██████████████████████| 1000/1000 [00:00<00:00, 122730.18it/s]


Done.

 iteration  exploitability
         0             0.0
       100             0.0
       200             0.0
       300             0.0
       400             0.0
       500             0.0
       600             0.0
       700             0.0
       800             0.0
       900             0.0
       999             0.0


Player 1 strategy (avg-iterate):
Infoset Action Probs
pl1_0__   [0.5, 0.5]

Player 2 strategy (avg-iterate):
Infoset Action Probs
pl2_0__   [0.5, 0.5]

P2 has ONE infoset — same [0.5, 0.5] mix regardless of what P1 did.
